# TRPG Runtime 调试 Notebook

这个 notebook 用来测试当前四模块最小运行时，并查看各阶段的中间结果。

当前链路是：

1. `Dicer` 与 `NPC Manager` 并行执行
2. `Narrator` 使用本轮 `Dicer/NPC` 结果 + 上一轮已经写回的 `Director state`
3. 当前轮结束后再生成下一轮的 `Director result`

说明：运行 `generate_opening()`、`trace_turn()`、`run_turn()` 或 `collect_director_update(wait=True)` 时，会真实调用当前配置的文本模型。

## Block 1: 初始化环境与辅助函数

这一格负责定位仓库根目录、导入运行时模块，并定义更易读的展示函数。

现在的 `show_json()` 不再只打印原始 JSON，而是会：

- 优先把字典渲染成键值表格
- 把标量列表渲染成项目列表
- 把字典列表渲染成表格
- 把原始 JSON 折叠到 `details` 里，方便需要时再展开

In [ ]:
from __future__ import annotations

import importlib
import inspect
import json
import sys
from html import escape
from pathlib import Path

from IPython.display import HTML, Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'Code').exists() and (candidate / 'Prompt').exists():
            return candidate
    raise RuntimeError(f'无法从当前路径定位仓库根目录: {start}')


ROOT = find_repo_root(Path.cwd().resolve())
CODE_DIR = (ROOT / 'Code').resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import trpg_runtime
import trpg_runtime.engine as trpg_runtime_engine
import trpg_runtime.models as trpg_runtime_models
import trpg_runtime.prompt_loader as trpg_runtime_prompt_loader
import trpg_runtime.state_store as trpg_runtime_state_store
import trpg_runtime.structured_output as trpg_runtime_structured_output

importlib.reload(trpg_runtime_models)
importlib.reload(trpg_runtime_prompt_loader)
importlib.reload(trpg_runtime_state_store)
importlib.reload(trpg_runtime_structured_output)
importlib.reload(trpg_runtime_engine)
importlib.reload(trpg_runtime)

from trpg_runtime import MinimalTRPGEngine, PromptRepository, parse_player_action


THEME_TEXT = 'var(--jp-content-font-color1, #e5e7eb)'
THEME_MUTED = 'var(--jp-ui-font-color2, #94a3b8)'
THEME_PANEL = 'var(--jp-layout-color1, #111827)'
THEME_PANEL_ALT = 'var(--jp-layout-color2, #1f2937)'
THEME_PANEL_SOFT = 'var(--jp-layout-color3, #0f172a)'
THEME_BORDER = 'var(--jp-border-color2, #475569)'
THEME_ACCENT = 'var(--jp-brand-color1, #60a5fa)'


def to_plain_data(value):
    if hasattr(value, 'model_dump'):
        return to_plain_data(value.model_dump(mode='python'))
    if isinstance(value, dict):
        return {key: to_plain_data(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_plain_data(item) for item in value]
    return value


def _is_scalar(value) -> bool:
    return value is None or isinstance(value, (str, int, float, bool))


def _scalar_text(value) -> str:
    if value is None:
        return 'null'
    if isinstance(value, bool):
        return 'true' if value else 'false'
    return str(value)


def _scalar_html(value) -> str:
    text = escape(_scalar_text(value))
    if value is None:
        return f"<span style='color:{THEME_MUTED}; font-style:italic;'>null</span>"
    return text


def _json_block(value) -> str:
    raw = escape(json.dumps(value, ensure_ascii=False, indent=2))
    return (
        f"<pre style='margin:8px 0 0 0; white-space:pre-wrap; word-break:break-word; color:{THEME_TEXT}; "
        f"background:{THEME_PANEL_ALT}; border:1px solid {THEME_BORDER}; padding:10px; border-radius:8px;'>"
        f"{raw}</pre>"
    )


def _key_value_table(rows: list[tuple[str, str]]) -> str:
    body = ''.join(
        f"<tr><th style='text-align:left; vertical-align:top; width:220px; color:{THEME_TEXT}; border:1px solid {THEME_BORDER}; padding:8px; background:{THEME_PANEL_ALT};'>{escape(label)}</th>"
        f"<td style='vertical-align:top; color:{THEME_TEXT}; border:1px solid {THEME_BORDER}; padding:8px; word-break:break-word; background:{THEME_PANEL};'>{value_html}</td></tr>"
        for label, value_html in rows
    )
    return (
        f"<table style='border-collapse:collapse; width:100%; margin:6px 0 14px 0; font-size:14px; color:{THEME_TEXT};'>"
        f"<tbody>{body}</tbody></table>"
    )


def _dict_list_table(items: list[dict], depth: int) -> str:
    columns: list[str] = []
    for item in items:
        for key in item.keys():
            if key not in columns:
                columns.append(key)

    header = ''.join(
        f"<th style='text-align:left; color:{THEME_TEXT}; border:1px solid {THEME_BORDER}; padding:8px; background:{THEME_PANEL_ALT};'>{escape(str(column))}</th>"
        for column in columns
    )
    rows = []
    for item in items:
        cells = ''.join(
            f"<td style='vertical-align:top; color:{THEME_TEXT}; border:1px solid {THEME_BORDER}; padding:8px; word-break:break-word; background:{THEME_PANEL};'>{_render_html(item.get(column), depth + 1)}</td>"
            for column in columns
        )
        rows.append(f'<tr>{cells}</tr>')
    return (
        f"<table style='border-collapse:collapse; width:100%; margin:6px 0 14px 0; font-size:14px; color:{THEME_TEXT};'>"
        f"<thead><tr>{header}</tr></thead><tbody>{''.join(rows)}</tbody></table>"
    )


def _render_html(value, depth: int = 0) -> str:
    value = to_plain_data(value)
    if _is_scalar(value):
        return _scalar_html(value)

    if depth >= 2:
        return _json_block(value)

    if isinstance(value, list):
        if not value:
            return f"<span style='color:{THEME_MUTED};'>empty list</span>"
        if all(_is_scalar(item) for item in value):
            items = ''.join(f"<li style='margin:2px 0;'>{_scalar_html(item)}</li>" for item in value)
            return f"<ul style='margin:0; padding-left:20px;'>{items}</ul>"
        if all(isinstance(item, dict) for item in value):
            return _dict_list_table(value, depth)
        items = ''.join(f"<li style='margin:6px 0;'>{_render_html(item, depth + 1)}</li>" for item in value)
        return f"<ol style='margin:0; padding-left:20px;'>{items}</ol>"

    if isinstance(value, dict):
        rows = [(str(key), _render_html(item, depth + 1)) for key, item in value.items()]
        return _key_value_table(rows)

    return escape(str(value))


def show_json(title: str, value, *, show_raw: bool = True) -> None:
    plain_value = to_plain_data(value)
    display(Markdown(f'### {title}'))
    display(HTML(f"<div style='color:{THEME_TEXT};'>{_render_html(plain_value)}</div>"))
    if show_raw:
        display(
            HTML(
                f"<details style='margin:8px 0 16px 0; color:{THEME_TEXT};'>"
                f"<summary style='cursor:pointer; color:{THEME_ACCENT};'>查看原始 JSON</summary>"
                f"{_json_block(plain_value)}"
                "</details>"
            )
        )


print('仓库根目录 =', ROOT)
print('代码目录 =', CODE_DIR)
print('MinimalTRPGEngine 来源 =', inspect.getfile(MinimalTRPGEngine))
print('from_prompt_files 签名 =', inspect.signature(MinimalTRPGEngine.from_prompt_files))


## Block 2: 配置测试参数

这一格用来切换规则、剧本、玩家名、玩家输入，以及当前场景里的 NPC / 物体 / 风险因素。

如果你想给 NPC 更明确的人设，也可以在 `NPC_STATES` 里补充说明；如果留空，系统会用最小默认信息创建 NPC。

In [ ]:
RULE_CODE = 'VHS'
STORY_CODE = 'THE_ANGEL'
PLAYER_NAME = '调查员'
PLAYER_INPUT = '我举着提灯进入中殿，先观察祭坛和玫瑰窗方向。'

VISIBLE_NPCS = ['卢卡斯']
INTERACTIVE_OBJECTS = ['祭坛', '玫瑰窗', '忏悔室木门']
HAZARDS = ['午夜后威胁升级']

NPC_STATES = {
    '卢卡斯': {
        'description': '教堂唯一幸存修士，信仰动摇但仍想阻止更大的灾难。',
        'attitude': '紧张而克制',
        'current_goal': '阻止外来者鲁莽深入地下墓室',
        'task_summary': '守在中殿附近观察异动',
        'last_public_status': '神色憔悴，低声祷告',
    }
}


## Block 3: 查看可用规则与剧本

这一格会读取 `Prompt/Rule` 和 `Prompt/Story`，确认你当前能选哪些规则和故事。

In [ ]:
repo = PromptRepository()
print('可用规则 =', repo.list_rule_codes())
print(f'{RULE_CODE} 可用剧本 =', repo.list_story_codes(RULE_CODE))


## Block 4: 创建引擎并查看初始状态

这一格会创建 `MinimalTRPGEngine`，把规则、剧本和场景初始信息装进去。

同时展示：

- 初始 `state`
- 玩家输入解析后的 `action`

In [ ]:
engine = MinimalTRPGEngine.from_prompt_files(
    rule_code=RULE_CODE,
    story_code=STORY_CODE,
    player_name=PLAYER_NAME,
    visible_npcs=VISIBLE_NPCS,
    interactive_objects=INTERACTIVE_OBJECTS,
    hazards=HAZARDS,
    npc_states=NPC_STATES,
)

show_json('初始状态 state', engine.state)
show_json('玩家输入解析 action', parse_player_action(PLAYER_INPUT))


## Block 5: 生成开场文案

这一格会调用 `engine.generate_opening()`，根据规则文本、剧本文本和 `BEGINNING_PROMPT.txt` 生成一段适合作为开场展示给玩家的介绍文案。

In [ ]:
opening = engine.generate_opening()
display(Markdown('### Opening 输出'))
print(opening)


## Block 6: 生成调试快照 `debug_trace`

`debug_trace = engine.trace_turn(PLAYER_INPUT)` 的意思是：

- 以当前 `engine.state` 为起点模拟一整轮
- `Dicer` 与 `NPC Manager` 并行运行
- `Narrator` 使用的是当前 state 里的旧 `director_state`
- 然后额外算出一个 `next_director_result`，表示“这一轮结束后给下一轮准备的 Director 结果”
- **不会修改 `engine.state`**

所以它是完整的调试快照，最适合排查时序问题。

In [ ]:
debug_trace = engine.trace_turn(PLAYER_INPUT)
debug_trace


## Block 7: 展开查看 `debug_trace` 的各部分

这一格会把调试快照拆开，逐项展示。

重点关注：

- `director_state_used`：Narrator 本轮真正参考的是它
- `next_director_context` / `next_director_result`：这是本轮结束后为下一轮准备的 Director 产物

In [ ]:
show_json('Director state used by narrator', debug_trace.director_state_used)
show_json('Dicer context', debug_trace.dicer_context)
show_json('Dicer output', debug_trace.dicer_result)
show_json('State before turn', debug_trace.state_before_turn)
show_json('State after dicer', debug_trace.state_after_dicer)
show_json('NPC context', debug_trace.npc_context)
show_json('NPC output', debug_trace.npc_result)
show_json('State after npc', debug_trace.state_after_npc)
show_json('Narrator context', debug_trace.narrator_context)
display(Markdown('### Narrator 输出'))
print(debug_trace.narration)
show_json('State after turn (before next director applied)', debug_trace.state_after_turn)
show_json('Next Director context', debug_trace.next_director_context)
show_json('Next Director result', debug_trace.next_director_result)
show_json('State after next director', debug_trace.state_after_next_director)


## Block 8: 正式提交这一回合

这一格会调用 `engine.run_turn(PLAYER_INPUT)`。

默认逻辑是：

- 本轮 Narrator 立即输出
- 然后把下一轮 Director 更新放进后台
- Director 在后台完成后会自动尝试写回 `engine.state`
- 所以返回值里的 `state` 仍然是“函数返回当下、Director 尚未写回”的状态快照

In [ ]:
result = engine.run_turn(PLAYER_INPUT)
display(Markdown('### 提交后的 Narrator 输出'))
print(result.narration)
show_json('提交后的 state (director 尚未写回)', result.state)
show_json('本轮 NPC output', result.npc_result)
show_json('本轮使用的 director_state', result.director_state_used)
print('director 后台任务是否已启动 =', result.pending_director_started)


## Block 9: 查看后台 Director 的最新结果与写回状态

这一格会阻塞等待后台 Director 完成，并返回最近一次 Director 结果。

如果后台 Director 已经在用户空档期自动写回了，这里通常会直接拿到那次结果；如果还没完成，这里会等它完成。

In [ ]:
director_update = engine.collect_director_update(wait=True)
show_json('后台 Director result', director_update)
show_json('Director 写回后的 engine.state', engine.state)
